# 실험 5. 앙상블 (LightGBM + XGBoost)

- 기준: `exp04`의 최적 LightGBM (4-6, OOF AUC 0.95909, 리더보드 0.95853)
- 방법: 같은 Fold로 XGBoost를 학습하고, OOF 예측을 섞어 AUC가 오르는지 확인

## 환경설정

In [1]:
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
START_TIME = time.time()

## 데이터 경로 탐색

In [2]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)

# 크로스플랫폼 경로 자동 해결

def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))  # drop duplicates, keep order

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.\n"
            "- Kaggle: 오른쪽 Input 패널에서 대회 데이터를 추가하세요.\n"
            "- 로컬: 노트북과 같은 위치의 data/ 폴더에 파일을 넣으세요."
        )
    raise RuntimeError(
        "데이터 폴더 후보가 여러 개입니다. 사용할 폴더를 DATA_DIR에 직접 지정하세요:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

DATA_DIR = /Users/youngjin/ai/kaggle_practice/Kaggle_Airline_Satisfaction/data


## 데이터 로드 및 전처리

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

TARGET = "satisfaction"
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]

In [4]:
y = train[TARGET].astype(int)
X = train.drop(columns=["id", TARGET]).copy()
X_test = test.drop(columns=["id"]).copy()

for col in CAT_COLS:
    categories = sorted(train[col].unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)

assert list(X.columns) == list(X_test.columns)
print("X      :", X.shape)
print("X_test :", X_test.shape)
print("y mean :", round(y.mean(), 4))

X      : (699635, 21)
X_test : (299844, 21)
y mean : 0.4436


## 교차검증 함수

In [5]:
LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.6,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 30,
    "seed": SEED,
    "verbose": -1,
}  # best setting from exp04 (4-6)
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def run_cv_lgb(X, X_test, params=LGB_PARAMS):
    """Stratified 5-fold LightGBM; returns OOF predictions and the averaged test prediction."""
    oof_pred = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
        dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

        oof_pred[va_idx] = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
        test_pred += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS
        print(f"Fold {fold}: AUC = {roc_auc_score(y.iloc[va_idx], oof_pred[va_idx]):.5f}   trees = {model.best_iteration}")

    return oof_pred, test_pred

## 기준 모델: LightGBM (실험 4-6 설정)

In [6]:
oof_lgb, test_lgb = run_cv_lgb(X, X_test)
auc_lgb = roc_auc_score(y, oof_lgb)
results = {"LightGBM": auc_lgb}
print(f"\nLightGBM OOF AUC = {auc_lgb:.5f}")

Fold 1: AUC = 0.95932   trees = 781


Fold 2: AUC = 0.95788   trees = 778


Fold 3: AUC = 0.95972   trees = 564


Fold 4: AUC = 0.95926   trees = 735


Fold 5: AUC = 0.95934   trees = 388

LightGBM OOF AUC = 0.95909


## 실험 5-1: XGBoost 5-Fold

- 이유: 같은 계열(GBDT)이지만 트리 성장 방식이 달라(depth-wise) 틀리는 샘플이 다를 수 있음. 범주형은 `enable_categorical`로 그대로 사용

In [7]:
XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "eta": 0.05,
    "max_depth": 7,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "reg_lambda": 30,
    "tree_method": "hist",
    "seed": SEED,
}


def run_cv_xgb(X, X_test, params=XGB_PARAMS):
    """Stratified 5-fold XGBoost with native categorical support; same folds as LightGBM."""
    oof_pred = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    dtest = xgb.DMatrix(X_test, enable_categorical=True)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        dtrain = xgb.DMatrix(X.iloc[tr_idx], label=y.iloc[tr_idx], enable_categorical=True)
        dvalid = xgb.DMatrix(X.iloc[va_idx], label=y.iloc[va_idx], enable_categorical=True)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            evals=[(dvalid, "valid")],
            early_stopping_rounds=100,
            verbose_eval=False,
        )

        best = (0, model.best_iteration + 1)
        oof_pred[va_idx] = model.predict(dvalid, iteration_range=best)
        test_pred += model.predict(dtest, iteration_range=best) / N_FOLDS
        print(f"Fold {fold}: AUC = {roc_auc_score(y.iloc[va_idx], oof_pred[va_idx]):.5f}   trees = {model.best_iteration + 1}")

    return oof_pred, test_pred

In [8]:
oof_xgb, test_xgb = run_cv_xgb(X, X_test)
auc_xgb = roc_auc_score(y, oof_xgb)
results["XGBoost"] = auc_xgb
print(f"\nXGBoost OOF AUC = {auc_xgb:.5f}   (LightGBM 대비 {auc_xgb - auc_lgb:+.5f})")

Fold 1: AUC = 0.95900   trees = 944


Fold 2: AUC = 0.95777   trees = 1085


Fold 3: AUC = 0.95973   trees = 766


Fold 4: AUC = 0.95891   trees = 1040


Fold 5: AUC = 0.95908   trees = 783

XGBoost OOF AUC = 0.95889   (LightGBM 대비 -0.00020)


## 실험 5-2: LightGBM + XGBoost 평균 블렌딩

- 이유: 두 모델의 오류가 다르면 평균이 단일 모델보다 나아짐. 예측 상관이 낮을수록 효과가 큼

In [9]:
oof_blend = (oof_lgb + oof_xgb) / 2
auc_blend = roc_auc_score(y, oof_blend)
results["LGB + XGB 평균"] = auc_blend
print(f"Blend OOF AUC = {auc_blend:.5f}   (LightGBM 대비 {auc_blend - auc_lgb:+.5f})")
print(f"OOF 예측 상관: {np.corrcoef(oof_lgb, oof_xgb)[0, 1]:.4f}")

Blend OOF AUC = 0.95911   (LightGBM 대비 +0.00002)
OOF 예측 상관: 0.9992
